### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors think bigger model = more compute. Seniors understand that MoE decouples **model capacity** (total parameters) from **inference cost** (active parameters per token). This is the key architectural insight that makes trillion-parameter models economically viable: you pay the memory cost of storing all experts, but only the compute cost of the active ones.

**Mental Model:** Imagine a hospital with 8 specialist doctors (cardiologist, neurologist, etc.) but only 2 examine each patient. The hospital has the *capacity* of 8 doctors, but the *cost per visit* is only 2. MoE works identically: 8 expert FFN networks exist, but only the Top-2 process each token.

**Common Junior Pitfall:** Assuming that "more experts = better." In reality, without careful **load balancing**, the router degenerates into always sending tokens to the same 1-2 experts (representation collapse), effectively reducing a 1.8T model to a 280B model that wasted 6x the memory. The auxiliary balancing loss is not optional — it's essential.

---

<a id="1"></a>
## 1. Conditional Computation: Why Dense Scaling Hits a Wall

### The Dense Scaling Problem

In a standard (dense) Transformer, every token passes through every parameter:

$$\text{FLOPs}_{\text{dense}} \propto \text{Total Parameters} \times \text{Sequence Length}$$

Doubling the model size **doubles** the compute cost per token. This creates an economic ceiling: at some point, the GPU-hours required for training (and the latency for inference) become prohibitive.

### The Conditional Computation Insight

**Key Idea:** What if different parts of the network activate for different inputs? Instead of one monolithic FFN processing everything, we have $N$ **expert** FFN sub-networks, and a lightweight **gating network** (router) decides which expert(s) each token should use.

$$\text{FLOPs}_{\text{MoE}} \propto \underbrace{K}_{\text{active experts}} \times \underbrace{\frac{\text{Total Params}}{N}}_{\text{params per expert}} \times \text{Seq Length}$$

With $N=8$ experts and $K=2$ active, you get **4x fewer FLOPs** while maintaining the model's total parameter count (capacity).

### Parameters vs. FLOPs: The Decoupling

| Metric | Dense 70B | MoE 8×7B (Mixtral) |
|---|---|---|
| **Total Parameters** | 70B | 46.7B (8 experts × ~5.6B each + shared) |
| **Active Parameters/Token** | 70B | ~12.9B (Top-2 experts + shared layers) |
| **FLOPs per Token** | ~140 TFLOPs | ~26 TFLOPs |
| **Quality** | Llama 2 70B | ≈ Llama 2 70B (often surpasses) |
| **Inference Speed** | 1x (baseline) | ~3-4x faster |

### 🎓 Socratic Deep Check
> *"If MoE gives us more capacity for less compute, why not make K=1 (only one expert per token) for maximum efficiency? What information-theoretic problem arises when a single expert must handle a token in complete isolation?"*

<details><summary>👉 Answer</summary>
With K=1, there is zero redundancy in the routing decision. If the router makes a mistake, the token gets processed by a completely wrong expert with no fallback. K=2 provides a form of 'error correction' — even if the best expert is not selected, the second expert likely captures related knowledge. Additionally, K=1 creates sharper gradient signals that make training unstable, and the hard routing creates non-differentiable boundaries that are difficult to optimize through.
</details>

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1. Conditional Computation: Why Dense Scaling Hits a Wall](#1-conditional-computation-why-dense-scaling-hits-a-wall)
  * [The Dense Scaling Problem](#the-dense-scaling-problem)
  * [The Conditional Computation Insight](#the-conditional-computation-insight)
  * [Parameters vs. FLOPs: The Decoupling](#parameters-vs-flops-the-decoupling)
  * [🎓 Socratic Deep Check](#-socratic-deep-check)
* [2. The MoE Architecture: Replacing the FFN](#2-the-moe-architecture-replacing-the-ffn)
  * [Where MoE Fits in a Transformer](#where-moe-fits-in-a-transformer)
  * [The Expert Network](#the-expert-network)
  * [🎓 Socratic Deep Check](#-socratic-deep-check)
* [3. Sparse Gating & Top-K Routing](#3-sparse-gating-top-k-routing)
  * [The Gating (Router) Network](#the-gating-router-network)
  * [Top-K Sparsification](#top-k-sparsification)
  * [The Complete Routing Algorithm](#the-complete-routing-algorithm)
  * [Noisy Top-K Gating (Shazeer et al., 2017)](#noisy-top-k-gating-shazeer-et-al-2017)
* [4. The Load Balancing Problem: Why Naive Routing Fails](#4-the-load-balancing-problem-why-naive-routing-fails)
  * [Representation Collapse](#representation-collapse)
  * [The Auxiliary Load Balancing Loss](#the-auxiliary-load-balancing-loss)
  * [Why $f_i \cdot p_i$?](#why-fi-cdot-pi)
  * [🎓 Socratic Deep Check](#-socratic-deep-check)
* [5. From-Scratch Implementation: A Complete MoE Layer](#5-from-scratch-implementation-a-complete-moe-layer)
  * [Architecture Overview](#architecture-overview)
* [6. End-to-End Demonstration: Routing Visualization](#6-end-to-end-demonstration-routing-visualization)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: Why does MoE replace the FFN but not the Attention layer?](#q1-why-does-moe-replace-the-ffn-but-not-the-attention-layer)
  * [Q2: What is representation collapse, and how does the auxiliary loss prevent it?](#q2-what-is-representation-collapse-and-how-does-the-auxiliary-loss-prevent-it)
  * [Q3: Why use noisy gating during training?](#q3-why-use-noisy-gating-during-training)
* [🔨 Practical Practice](#-practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)

---


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# VISUALIZATION: Dense vs. MoE Scaling
# ============================================================
total_params = np.array([7, 13, 30, 70, 180, 540])  # Billions
num_experts = 8
top_k = 2

dense_flops = total_params * 2   # ~2 FLOPs per param per token (forward pass)
moe_flops = (total_params / num_experts) * top_k * 2  # Only K experts fire

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(total_params, dense_flops, 'o-', color='#E53935', lw=2.5, ms=8, label='Dense (all params active)')
ax.plot(total_params, moe_flops, 's-', color='#43A047', lw=2.5, ms=8, label=f'MoE (Top-{top_k} of {num_experts} experts)')
ax.fill_between(total_params, moe_flops, dense_flops, alpha=0.15, color='#43A047')
ax.annotate(f'{dense_flops[-1]/moe_flops[-1]:.0f}x savings\nat 540B params',
            xy=(540, moe_flops[-1]), xytext=(350, 500),
            fontsize=12, fontweight='bold', color='#2E7D32',
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.set_xlabel('Total Model Parameters (Billions)', fontsize=13)
ax.set_ylabel('FLOPs per Token (Billions)', fontsize=13)
ax.set_title('The MoE Scaling Advantage: Parameters vs. Compute', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

print(f"At 540B parameters:")
print(f"  Dense FLOPs/token:  {dense_flops[-1]:.0f}B")
print(f"  MoE FLOPs/token:    {moe_flops[-1]:.0f}B")
print(f"  Compute reduction:  {dense_flops[-1]/moe_flops[-1]:.1f}x")

<a id="2"></a>
## 2. The MoE Architecture: Replacing the FFN

### Where MoE Fits in a Transformer

Recall from Module 06 that each Transformer block has two sub-layers:
1. **Multi-Head Attention** (shared across all tokens — not modified)
2. **Feed-Forward Network (FFN)** — this is what MoE replaces

In a standard Transformer:
$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

In an MoE Transformer, this single FFN is replaced by $N$ parallel expert FFNs and a gating router:
$$\text{MoE}(x) = \sum_{i=1}^{N} g_i(x) \cdot E_i(x)$$

Where:
- $E_i(x)$ is the output of expert $i$ (each expert is a full FFN)
- $g_i(x)$ is the **gating weight** for expert $i$ (output of the router)
- In sparse MoE, most $g_i(x) = 0$ — only the Top-K experts have non-zero gates

### The Expert Network
Each expert $E_i$ is an independent FFN with its own weights:
$$E_i(x) = W_2^{(i)} \cdot \text{SiLU}(W_1^{(i)} x) \odot (W_3^{(i)} x)$$

Modern MoE models (Mixtral) use the **SwiGLU** activation (gated linear unit with SiLU), which is the same FFN used in Llama-style dense models.

### 🎓 Socratic Deep Check
> *"The attention layers are SHARED across all tokens (not duplicated per expert). Why? What would happen if we also made attention expert-specific?"*

<details><summary>👉 Answer</summary>
Attention is a global token-mixing operation — it needs to see ALL tokens to compute relevance scores. Making attention expert-specific would mean each expert only attends to its own subset of tokens, destroying the model's ability to capture long-range dependencies. The FFN, by contrast, operates token-independently (each token processed separately), making it a natural candidate for expert specialization without losing cross-token information.
</details>

📈 **Production Signal:** In Mixtral 8x7B, every other Transformer layer is an MoE layer (alternating dense/MoE). DBRX uses MoE at every layer. The architectural choice of "which layers get experts" directly impacts the quality-compute tradeoff.

<a id="3"></a>
## 3. Sparse Gating & Top-K Routing

### The Gating (Router) Network

The router is a simple linear layer that maps each token's hidden state to a probability distribution over $N$ experts:

$$G(x) = \text{Softmax}(W_g \cdot x)$$

Where $W_g \in \mathbb{R}^{N \times d_{\text{model}}}$ is the router's weight matrix.

### Top-K Sparsification

To enforce sparsity, we zero out all but the top-$K$ values **before** the softmax:

$$g(x) = \text{Softmax}\left(\text{TopK}(W_g \cdot x, K)\right)$$

Where $\text{TopK}(v, K)$ keeps the $K$ largest entries of $v$ and sets the rest to $-\infty$:

$$\text{TopK}(v, K)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the top-K values} \\ -\infty & \text{otherwise} \end{cases}$$

After softmax, the $-\infty$ entries become exactly 0, ensuring only $K$ experts are activated.

### The Complete Routing Algorithm

For a single token $x$:

1. **Compute logits:** $\ell = W_g \cdot x \in \mathbb{R}^N$
2. **Select top-K:** Find indices $\{i_1, \ldots, i_K\}$ of the $K$ largest $\ell_i$
3. **Compute gates:** $g_{i_k} = \frac{\exp(\ell_{i_k})}{\sum_{j=1}^{K} \exp(\ell_{i_j})}$ for $k = 1, \ldots, K$
4. **Compute output:** $y = \sum_{k=1}^{K} g_{i_k} \cdot E_{i_k}(x)$

### Noisy Top-K Gating (Shazeer et al., 2017)

To encourage exploration during training, Gaussian noise is added to the router logits before the Top-K selection:

$$\ell_{\text{noisy}} = \ell + \epsilon \cdot \text{Softplus}(W_{\text{noise}} \cdot x), \quad \epsilon \sim \mathcal{N}(0, 1)$$

This prevents the router from prematurely committing to a fixed routing pattern and helps discover better expert assignments.

📈 **Production Signal:** Mixtral uses Top-2 routing with 8 experts. Switch Transformer (Google) uses Top-1 routing with 128+ experts for maximum throughput at extreme scale. The choice is a latency vs. quality trade-off.

In [ ]:
# ============================================================
# DEMONSTRATION: Top-K Sparse Gating Mechanics
# ============================================================
torch.manual_seed(42)

d_model = 16
num_experts = 8
top_k = 2

# Simulated router
W_gate = torch.randn(num_experts, d_model) * 0.1
x = torch.randn(d_model)  # A single token embedding

# Step 1: Raw logits
logits = W_gate @ x
print("Step 1 — Raw router logits (one score per expert):")
for i, l in enumerate(logits):
    print(f"  Expert {i}: {l.item():+.4f}")

# Step 2: Top-K selection
top_k_values, top_k_indices = torch.topk(logits, top_k)
print(f"\nStep 2 — Top-{top_k} experts selected: {top_k_indices.tolist()}")
print(f"          Their logits: {top_k_values.tolist()}")

# Step 3: Sparse softmax (only over selected experts)
sparse_gates = F.softmax(top_k_values, dim=0)
print(f"\nStep 3 — Gating weights (softmax over Top-{top_k} only):")
for idx, gate in zip(top_k_indices, sparse_gates):
    print(f"  Expert {idx.item()}: {gate.item():.4f}")
print(f"  Sum: {sparse_gates.sum().item():.4f} (should be 1.0)")
print(f"  Other 6 experts: gate = 0.0 (not computed at all)")

"""
What just happened?
We demonstrated the core routing mechanism. Out of 8 experts, only 2 were
selected. The softmax is computed ONLY over the top-2 logits, so the gates
sum to 1.0 across the active experts. The other 6 experts contribute nothing
— their FFN forward pass is never even executed, saving 75% of compute.
"""

<a id="4"></a>
## 4. The Load Balancing Problem: Why Naive Routing Fails

### Representation Collapse

Left to its own devices, the router quickly degenerates into a pathological state: it routes **all tokens to the same 1-2 experts**, while the remaining experts receive zero gradient updates and become dead weight.

**Why does this happen?** It's a positive feedback loop:
1. Expert $i$ receives slightly more tokens → gets more gradient updates → improves slightly
2. The router sees Expert $i$ is better → sends even MORE tokens to it
3. Other experts get fewer tokens → fewer updates → fall further behind
4. Eventually: 1 expert handles everything, 7 experts are useless

This is **representation collapse**, also called the "winner-take-all" problem. It transforms a 46.7B MoE model into an expensive 5.6B single-expert model.

### The Auxiliary Load Balancing Loss

To prevent collapse, we add a regularization term to the training loss that penalizes uneven expert utilization.

**For a batch of $T$ tokens routed across $N$ experts:**

1. **Fraction of tokens dispatched to expert $i$:**
$$f_i = \frac{1}{T} \sum_{t=1}^{T} \mathbb{1}[i \in \text{TopK}(G(x_t))]$$

2. **Average router probability for expert $i$:**
$$p_i = \frac{1}{T} \sum_{t=1}^{T} \text{Softmax}(W_g \cdot x_t)_i$$

3. **Load Balancing Loss:**
$$\mathcal{L}_{\text{balance}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

Where $\alpha$ is a small coefficient (typically 0.01).

### Why $f_i \cdot p_i$?

- $f_i$ measures actual usage (binary: was expert selected?)
- $p_i$ measures the router's continuous probability (differentiable!)
- Their product is minimized when both are uniform: $f_i = p_i = 1/N$ for all $i$
- If one expert gets 50% of tokens ($f_i = 0.5$) while others get $\sim$7% each, the $f_i \cdot p_i$ product is large for that expert, and the gradient pushes the router to redistribute

The factor $N$ ensures the loss is properly scaled regardless of the number of experts.

### 🎓 Socratic Deep Check
> *"Why can't we just use $f_i$ alone for the balancing loss? Why do we need the product $f_i \cdot p_i$?"*

<details><summary>👉 Answer</summary>
$f_i$ involves the TopK operation, which is non-differentiable (it's a discrete selection). We cannot backpropagate through argmax. By multiplying with $p_i$ (which IS differentiable — it's a softmax output), we create a loss that has useful gradients. The gradient flows through $p_i$ to update the router weights $W_g$, while $f_i$ acts as a weighting factor that identifies which experts are overloaded.
</details>

📈 **Production Signal:** The balancing coefficient $\alpha$ is a critical hyperparameter. Too small: experts collapse. Too large: the model prioritizes balance over quality, and all experts become identical (losing the benefit of specialization). Mixtral uses $\alpha = 0.01$.

In [ ]:
# ============================================================
# DEMONSTRATION: Load Balancing Loss Mechanics
# ============================================================
torch.manual_seed(42)

N_EXPERTS = 8
T_TOKENS = 100
D_MODEL = 16
TOP_K = 2

# Simulate a batch of token embeddings
tokens = torch.randn(T_TOKENS, D_MODEL)
W_gate = torch.randn(N_EXPERTS, D_MODEL) * 0.1

# Compute router logits and probabilities
logits = tokens @ W_gate.T  # [T, N]
probs = F.softmax(logits, dim=-1)  # [T, N]

# Top-K selection
_, top_indices = torch.topk(logits, TOP_K, dim=-1)  # [T, K]

# f_i: fraction of tokens dispatched to each expert
expert_counts = torch.zeros(N_EXPERTS)
for idx in top_indices.flatten():
    expert_counts[idx] += 1
f_i = expert_counts / T_TOKENS  # Normalized

# p_i: average router probability for each expert
p_i = probs.mean(dim=0)

# Load balancing loss
alpha = 0.01
L_balance = alpha * N_EXPERTS * (f_i * p_i).sum()

print("Expert Load Distribution:")
print(f"{'Expert':<10} {'Tokens (f_i)':<15} {'Avg Prob (p_i)':<18} {'f_i × p_i':<12}")
print("-" * 55)
for i in range(N_EXPERTS):
    bar = '█' * int(f_i[i].item() * 40)
    print(f"  E{i:<7} {f_i[i]:.4f}         {p_i[i]:.4f}            {(f_i[i]*p_i[i]):.6f}  {bar}")

ideal = 1.0 / N_EXPERTS
print(f"\nIdeal f_i = {ideal:.4f} (uniform)")
print(f"Load Balancing Loss: {L_balance.item():.6f}")
print(f"Minimum possible (perfect balance): {alpha * N_EXPERTS * N_EXPERTS * (ideal * ideal):.6f}")

"""
What just happened?
We computed the load balancing loss for a batch of tokens. Notice how some
experts receive disproportionately more tokens than others. The auxiliary loss
penalizes this imbalance — its gradient will push the router to distribute
tokens more evenly, preventing the winner-take-all collapse.
"""

<a id="5"></a>
## 5. From-Scratch Implementation: A Complete MoE Layer

Now we bring everything together into a functional, trainable MoE layer that could be dropped into any Transformer architecture.

### Architecture Overview

```
Input Token x ∈ ℝ^d
       │
       ├──► Router(x) = Softmax(TopK(Wg·x + noise))
       │       │
       │       ├──► gate_1, expert_idx_1
       │       └──► gate_2, expert_idx_2
       │
       ├──► Expert_idx_1(x) × gate_1  ──┐
       └──► Expert_idx_2(x) × gate_2  ──┼──► Output y ∈ ℝ^d
                                        │
                              (weighted sum)
```

📈 **Production Signal:** In production MoE systems (like Megablocks or vLLM), tokens are physically **re-sorted** by expert assignment and processed in batches — this maximizes GPU utilization by turning sparse routing into dense, contiguous matrix multiplications.

In [ ]:
# ============================================================
# COMPLETE MoE LAYER — From Scratch in PyTorch
# ============================================================

class Expert(nn.Module):
    """A single expert: a standard Feed-Forward Network (SwiGLU variant)."""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)  # Gate projection for SwiGLU
    
    def forward(self, x):
        # SwiGLU: (SiLU(W1·x) ⊙ W3·x) → W2
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class NoisyTopKRouter(nn.Module):
    """Sparse gating network with noisy Top-K routing."""
    
    def __init__(self, d_model, num_experts, top_k, noise_std=0.1):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        self.noise_std = noise_std
        self.gate = nn.Linear(d_model, num_experts, bias=False)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, d_model]
        Returns:
            gates: [batch_size, seq_len, top_k] — gating weights
            indices: [batch_size, seq_len, top_k] — expert indices
            aux_loss: scalar — load balancing loss
        """
        # Raw logits
        logits = self.gate(x)  # [B, S, N_experts]
        
        # Add noise during training for exploration
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(logits) * self.noise_std
            logits = logits + noise
        
        # Compute full softmax for load balancing loss (before TopK)
        router_probs = F.softmax(logits, dim=-1)  # [B, S, N]
        
        # Top-K selection
        top_k_values, top_k_indices = torch.topk(logits, self.top_k, dim=-1)
        
        # Sparse softmax: normalize only over selected experts
        top_k_gates = F.softmax(top_k_values, dim=-1)  # [B, S, K]
        
        # --- Auxiliary Load Balancing Loss ---
        # f_i: fraction of tokens routed to each expert
        # (using one-hot encoding of selected experts)
        one_hot = F.one_hot(top_k_indices, self.num_experts).float()  # [B, S, K, N]
        tokens_per_expert = one_hot.sum(dim=2).mean(dim=[0, 1])  # [N] — avg across batch & seq
        
        # p_i: average router probability for each expert
        avg_probs = router_probs.mean(dim=[0, 1])  # [N]
        
        # L_balance = alpha * N * sum(f_i * p_i)
        aux_loss = self.num_experts * (tokens_per_expert * avg_probs).sum()
        
        return top_k_gates, top_k_indices, aux_loss


class MixtureOfExperts(nn.Module):
    """Complete MoE layer: replaces the standard FFN in a Transformer block."""
    
    def __init__(self, d_model, d_ff, num_experts=8, top_k=2, aux_loss_coef=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.aux_loss_coef = aux_loss_coef
        
        # N independent expert FFNs
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])
        
        # Router
        self.router = NoisyTopKRouter(d_model, num_experts, top_k)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, d_model]
        Returns:
            output: [batch_size, seq_len, d_model]
            aux_loss: scalar
        """
        B, S, D = x.shape
        
        # Route tokens to experts
        gates, indices, aux_loss = self.router(x)  # gates: [B,S,K], indices: [B,S,K]
        
        # Initialize output
        output = torch.zeros_like(x)  # [B, S, D]
        
        # Process each expert
        # (In production, tokens are batched per-expert for efficiency;
        #  here we loop for clarity)
        for k in range(self.top_k):
            expert_indices = indices[:, :, k]  # [B, S] — which expert for slot k
            expert_gates = gates[:, :, k].unsqueeze(-1)  # [B, S, 1] — gate weight
            
            for expert_id in range(self.num_experts):
                # Find all tokens assigned to this expert in this slot
                mask = (expert_indices == expert_id)  # [B, S]
                if mask.any():
                    # Get tokens for this expert
                    expert_input = x[mask]  # [num_tokens, D]
                    expert_output = self.experts[expert_id](expert_input)  # [num_tokens, D]
                    
                    # Weight by gate and accumulate
                    gate_values = expert_gates[mask]  # [num_tokens, 1]
                    output[mask] += gate_values * expert_output
        
        return output, self.aux_loss_coef * aux_loss


# --- Instantiate and test ---
torch.manual_seed(42)
D_MODEL = 64
D_FF = 128
NUM_EXPERTS = 8
TOP_K = 2

moe_layer = MixtureOfExperts(D_MODEL, D_FF, NUM_EXPERTS, TOP_K)

# Simulated input: batch=2, seq_len=10, d_model=64
x = torch.randn(2, 10, D_MODEL)
output, aux_loss = moe_layer(x)

print(f"Input shape:        {x.shape}")
print(f"Output shape:       {output.shape}")
print(f"Aux (balance) loss: {aux_loss.item():.6f}")

# Parameter count comparison
total_params = sum(p.numel() for p in moe_layer.parameters())
expert_params = sum(p.numel() for p in moe_layer.experts[0].parameters())
active_params = expert_params * TOP_K + sum(p.numel() for p in moe_layer.router.parameters())
print(f"\nTotal parameters:   {total_params:,}")
print(f"Active per token:   {active_params:,} ({active_params/total_params*100:.1f}%)")
print(f"Compute savings:    {total_params/active_params:.1f}x")

"""
What just happened?
We built a complete MoE layer from scratch. The key components:
1. Expert: A SwiGLU FFN (identical to what's in Llama/Mixtral)
2. NoisyTopKRouter: Computes sparse gates AND the auxiliary load balancing loss
3. MixtureOfExperts: Orchestrates routing, expert computation, and output recombination

Only 2 of 8 experts fire per token — meaning ~25% of FFN parameters are active,
while the model retains the full capacity of all 8 experts.
"""

<a id="6"></a>
## 6. End-to-End Demonstration: Routing Visualization

Let's visualize how the router distributes tokens across experts, demonstrating the effect of the load balancing loss.

In [ ]:
# ============================================================
# VISUALIZATION: Token-to-Expert Routing Heatmap
# ============================================================
torch.manual_seed(7)

# Create a batch of diverse tokens (simulating different "types")
seq_len = 20
x_demo = torch.randn(1, seq_len, D_MODEL)

moe_layer.eval()
with torch.no_grad():
    gates, indices, _ = moe_layer.router(x_demo)

# Build routing matrix: [seq_len, num_experts]
routing_map = torch.zeros(seq_len, NUM_EXPERTS)
for t in range(seq_len):
    for k in range(TOP_K):
        expert_id = indices[0, t, k].item()
        gate_val = gates[0, t, k].item()
        routing_map[t, expert_id] = gate_val

fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [3, 1]})

# Heatmap
im = axes[0].imshow(routing_map.numpy().T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
axes[0].set_xlabel('Token Position', fontsize=13)
axes[0].set_ylabel('Expert ID', fontsize=13)
axes[0].set_title('Token → Expert Routing Map (Gate Weights)', fontsize=14, fontweight='bold')
axes[0].set_yticks(range(NUM_EXPERTS))
axes[0].set_yticklabels([f'Expert {i}' for i in range(NUM_EXPERTS)])
plt.colorbar(im, ax=axes[0], label='Gate Weight', shrink=0.8)

# Expert load bar chart
expert_load = (routing_map > 0).float().sum(dim=0).numpy()
colors = ['#43A047' if abs(l - seq_len*TOP_K/NUM_EXPERTS) < 2 else '#E53935' for l in expert_load]
axes[1].barh(range(NUM_EXPERTS), expert_load, color=colors, edgecolor='white', linewidth=1.5)
axes[1].axvline(x=seq_len*TOP_K/NUM_EXPERTS, color='#1565C0', linestyle='--', lw=2, label=f'Ideal ({seq_len*TOP_K/NUM_EXPERTS:.0f})')
axes[1].set_xlabel('Tokens Received', fontsize=13)
axes[1].set_title('Expert Load', fontsize=14, fontweight='bold')
axes[1].set_yticks(range(NUM_EXPERTS))
axes[1].set_yticklabels([f'E{i}' for i in range(NUM_EXPERTS)])
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nWith perfect load balance, each expert receives {seq_len*TOP_K/NUM_EXPERTS:.0f} tokens.")
print(f"Actual distribution: {expert_load.tolist()}")
print(f"Max/Min ratio: {expert_load.max()/max(expert_load.min(), 1):.1f}x")

"""
What just happened?
We visualized the router's token assignments. Each row in the heatmap shows
which 2 experts (Top-K=2) were selected for each token, and with what weight.
The bar chart reveals whether load is balanced — green bars are near the ideal
line, red bars indicate imbalance that the auxiliary loss would correct.
"""

---
## ✅ Knowledge Check

### Q1: Why does MoE replace the FFN but not the Attention layer?
<details><summary>👉 Answer</summary>
The FFN operates independently on each token — making it trivially parallelizable across experts. Attention requires all tokens to interact (computing pairwise scores), so routing different tokens to different attention experts would break the global information exchange that makes attention powerful. The FFN is where the model stores factual knowledge; attention is where it routes information between tokens.
</details>

### Q2: What is representation collapse, and how does the auxiliary loss prevent it?
<details><summary>👉 Answer</summary>
Representation collapse is when the router converges on sending all tokens to 1-2 experts, making the other experts useless. This happens because better-trained experts attract more tokens, creating a rich-get-richer feedback loop. The auxiliary loss $\alpha \cdot N \cdot \sum f_i \cdot p_i$ penalizes uneven load distribution — its gradient pushes the router to spread tokens more uniformly across experts, breaking the positive feedback loop.
</details>

### Q3: Why use noisy gating during training?
<details><summary>👉 Answer</summary>
Without noise, the router's Top-K selection creates a discrete, deterministic mapping that can get stuck in local optima early in training. Adding Gaussian noise to the logits before TopK selection introduces stochastic exploration — a token that would normally go to Expert 3 might occasionally go to Expert 5, allowing Expert 5 to demonstrate its value. This is analogous to ε-greedy exploration in reinforcement learning.
</details>

---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Modify the `MixtureOfExperts` class to use `Top-1` routing instead of `Top-2`. Run inference and compare the output. How does the parameter efficiency change?
2. Remove the noise from `NoisyTopKRouter` (set `noise_std=0`). Run the routing visualization 5 times with different seeds. Does the routing become more or less diverse?

### Tier 2: Intermediate
1. **Expert Capacity:** Implement an expert capacity buffer — if an expert exceeds $C = \frac{K \cdot T}{N} \cdot \text{capacity\_factor}$ tokens, additional tokens are dropped (or sent to a second-choice expert). This is how Switch Transformer handles load imbalance at scale.
2. **Memory Math:** Calculate the total VRAM required for a Mixtral 8x7B model in float16. Compare it to a dense Llama 2 70B. Why does MoE require MORE memory despite using LESS compute? What does this imply for single-GPU deployment?

### Tier 3: Advanced
1. **MoE Transformer Block:** Integrate the `MixtureOfExperts` layer into the `TransformerBlock` from Module 06 (replacing the FFN). Build a 4-layer mini-MoE-Transformer and train it on a character-level text dataset. Compare training loss curves against an equivalent dense model with the same active parameter count.
2. **Expert Merging:** After training, compute the cosine similarity between all pairs of expert weight matrices. If two experts are >95% similar, they've failed to specialize. Implement a post-training expert merging strategy that reduces the expert count while preserving quality.